# Microsoft Stock Price Direction Classification — CSE 6011

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hhsonet/ML/blob/main/notebooks/msft-price-direction-classification.ipynb)

**Problem:** predict whether Microsoft's next trading-day adjusted close will move **Up** or **Down**.  
**Dataset:** historical MSFT OHLCV data downloaded at runtime with `yfinance`.  
**Goal:** compare multiple classifiers with a chronological train/test split.

This is an educational ML exercise, not a trading system or financial recommendation.

## 1. Load the data

The original notebook depended on a private Google Drive file. This version downloads a fixed historical period so anyone can reproduce the workflow in Colab or locally with internet access.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

df = yf.download(
    "MSFT",
    start="2000-01-01",
    end="2021-05-01",
    auto_adjust=False,
    progress=False,
)

# Recent yfinance versions can return a MultiIndex even for one ticker.
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

if "Adj Close" not in df.columns:
    df["Adj Close"] = df["Close"]

print("Dataset shape:", df.shape)
df.head()

## 2. Explore the data (EDA)

Create a next-day direction target and inspect its class distribution. Unlike the original notebook, the target is not overwritten by a global mean threshold.

In [ ]:
df = df.copy()
df["Return_1d"] = df["Adj Close"].pct_change()
df["Intraday_Range"] = (df["High"] - df["Low"]) / df["Open"]
df["Open_Close_Change"] = (df["Close"] - df["Open"]) / df["Open"]
df["Volume_Change"] = df["Volume"].pct_change()
df["MA5_Ratio"] = df["Close"] / df["Close"].rolling(5).mean() - 1
df["Price_Direction"] = np.where(df["Adj Close"].shift(-1) > df["Adj Close"], "Up", "Down")

FEATURES = [
    "Return_1d",
    "Intraday_Range",
    "Open_Close_Change",
    "Volume_Change",
    "MA5_Ratio",
]

model_data = (
    df[FEATURES + ["Price_Direction"]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .iloc[:-1]
)

print(model_data["Price_Direction"].value_counts())
model_data["Price_Direction"].value_counts().plot(
    kind="bar",
    title="Next-day price direction",
)
plt.xlabel("Direction")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

## 3. Preprocess the data

Because this is time-series data, the first 80% of observations are used for training and the final 20% for testing. Scaling is fitted only on the training period. SMOTE is also applied **only to training data**, preventing leakage from the test set.

In [ ]:
X = model_data[FEATURES]
y = model_data["Price_Direction"]

split_index = int(len(model_data) * 0.80)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print("Training period:", X_train.index.min(), "to", X_train.index.max())
print("Testing period:", X_test.index.min(), "to", X_test.index.max())
print("\nTraining classes before SMOTE:")
print(y_train.value_counts())
print("\nTraining classes after SMOTE:")
print(pd.Series(y_train_balanced).value_counts())

## 4. Train the models

Train several common classifiers on the same balanced training set.

In [ ]:
models = {
    "SVM": SVC(probability=True, random_state=42),
    "Gaussian NB": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=10),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=100,
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),
}

fitted_models = {}
rows = []

for name, model in models.items():
    model.fit(X_train_balanced, y_train_balanced)
    fitted_models[name] = model

    pred = model.predict(X_test_scaled)
    proba = model.predict_proba(X_test_scaled)[:, 1]
    classes = list(model.classes_)
    up_index = classes.index("Up")
    up_proba = model.predict_proba(X_test_scaled)[:, up_index]

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, pos_label="Up", zero_division=0),
        "Recall": recall_score(y_test, pred, pos_label="Up", zero_division=0),
        "F1": f1_score(y_test, pred, pos_label="Up", zero_division=0),
        "ROC_AUC": roc_auc_score((y_test == "Up").astype(int), up_proba),
    })

results = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
results

## 5. Evaluate the models

Use the comparison table as the primary result and inspect the best model's classification report and confusion matrix.

In [ ]:
best_name = results.loc[0, "Model"]
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test_scaled)

print("Best model by F1:", best_name)
print(classification_report(y_test, best_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_pred,
    labels=["Down", "Up"],
    display_labels=["Down", "Up"],
    cmap="Blues",
)
plt.title(f"Confusion Matrix — {best_name}")
plt.show()

## 6. Conclusion

This version removes the private Drive dependency and avoids the original SMOTE-before-split leakage. It also predicts the **next day's** direction rather than defining the label from the same day's price level.

Stock direction is noisy and non-stationary, so strong historical test metrics should not be interpreted as reliable future trading performance. A next step would be walk-forward validation with transaction-cost assumptions.